Default Switch からVMに接続し InternalSwitch に固定IPアドレスを設定する

# 定義

In [ ]:
$vmName           = 'Windows2025'
$internalSwitch   = 'InternalSwitch'
$vmInternalIP     = '192.168.128.10'
$internalSwitchIP = '192.168.128.254'
$prefixLength     = 24
$cred = New-Object System.Management.Automation.PSCredential (
    'Administrator',
    (ConvertTo-SecureString -AsPlainText -Force -String 'Passw0rd!')
)

# 内部スイッチが無ければ作る

In [ ]:
if (Get-VMSwitch -Name $internalSwitch -ErrorAction SilentlyContinue) {'OK'}
else {
    New-VMSwitch -Name $internalSwitch -SwitchType Internal -Verbose
    New-NetIPAddress -InterfaceAlias "vEthernet ($internalSwitch)" -IPAddress $internalSwitchIP -PrefixLength $prefixLength
}

# VMにスイッチを接続

In [ ]:
if (-not(Get-VMNetworkAdapter -VMName $vmName | Where-Object { $_.SwitchName -eq $internalSwitch })) {
    Add-VMNetworkAdapter -VMName $vmName -SwitchName $internalSwitch -Verbose
}
Get-VMNetworkAdapter -VMName $vmName

# WinRM 設定

In [ ]:
$winrm = Get-Service WinRM
if ($winrm.Status -eq 'Stopped' -and $winrm.StartType -eq 'Manual') {
    winrm quickconfig -force
}

# TrustedHosts 設定

In [ ]:
$current = (Get-Item WSMan:\localhost\Client\TrustedHosts).Value
$hostList = if ($current) { $current -split ',' | ForEach-Object { $_.Trim() } } else { @() }
if ($hostList -notcontains $dsIpAddress -and $current -ne '*') {
    $hostList = @($hostList) + @($dsIpAddress) -join ','
    Set-Item WSMan:\localhost\Client\TrustedHosts -Value $hostList -Force -Verbose
}
if ($hostList -notcontains $vmInternalIP -and $current -ne '*') {
    $hostList = @($hostList) + @($vmInternalIP) -join ','
    Set-Item WSMan:\localhost\Client\TrustedHosts -Value $hostList -Force -Verbose
}
Get-Item WSMan:\localhost\Client\TrustedHosts

# Default Switch のIPアドレスを取得

In [ ]:
$dsIpAddress = (Get-VMNetworkAdapter -VMName $vmName | Where-Object {$_.SwitchName -eq 'Default Switch'}).IPAddresses |
  Where-Object {($_ -match '^\d{1,3}(\.\d{1,3}){3}$') -and ($_ -notlike '169.254*') -and ($_ -ne '0.0.0.0')}
$dsIpAddress

# CimSessionを作成

In [ ]:
$cimSession = New-CimSession -ComputerName $dsIpAddress -Credential $cred -Verbose
$cimSession

# 固定IPアドレスを設定
リンクローカルアドレスになっているものにIPアドレスを設定

In [ ]:
Get-NetIPAddress -CimSession $cimSession -AddressFamily IPv4 -IPAddress '169.254.*' |
  New-NetIPAddress -IPAddress $vmInternalIP -PrefixLength $prefixLength -Verbose

# ネットワークプロファイルをプライベートにする

In [ ]:
Get-NetIPAddress -CimSession $cimSession -IPAddress $vmInternalIP |
  Set-NetConnectionProfile -NetworkCategory Private -Verbose -PassThru

# Cim Session 切断

In [ ]:
Remove-CimSession $cimSession

# 設定した固定IPの PS Session を作成

In [ ]:
$pss = New-PSSession -ComputerName $vmInternalIP -Credential $cred
       New-CimSession -ComputerName $dsIpAddress -Credential $cred -Verbose
$pss

# IP設定確認

In [ ]:
Invoke-Command -Session $pss -Command { Get-NetIPAddress -AddressFamily IPv4}

# PS Session 切断

In [ ]:
$pss | Remove-PSSession

# TrustedHosts 削除

In [ ]:
$current = (Get-Item WSMan:\localhost\Client\TrustedHosts).Value
if ($current -and $current -ne '*' -and $current -ne '') {
    $hostList = $current -split "," | ForEach-Object { $_.Trim() }
    if ($hostList -contains $dsIpAddress) {
        $hostList = ($hostList | Where-Object { $_ -ne $dsIpAddress }) -join ","
        Set-Item WSMan:\localhost\Client\TrustedHosts -Value $hostList -Force
    }
    if ($hostList -contains $vmInternalIP) {
        $hostList = ($hostList | Where-Object { $_ -ne $vmInternalIP }) -join ","
        Set-Item WSMan:\localhost\Client\TrustedHosts -Value $hostList -Force
    }
}
Get-Item WSMan:\localhost\Client\TrustedHosts